# RetailStream Inc.

## End-to-End Data Engineering Pipeline

### Batch • Incremental • Streaming • Delta Lake

Built using

✔ Databricks

✔ PySpark

✔ Spark SQL

✔ Delta Lake

✔ Auto Loader

✔ Unity Catalog

# Project Overview

RetailStream Inc. is a retail company that receives order and transaction data from multiple stores across different cities.

The objective of this project is to build a scalable end-to-end Data Engineering pipeline capable of processing:

- Historical Batch Data
- Incremental Batch Data
- Real-Time Streaming Data
- Late Arriving Data

The pipeline follows the Medallion Architecture (Bronze → Silver → Gold) using Delta Lake to produce analytics-ready datasets.

## Business Problem

RetailStream Inc. receives order data from multiple stores across India.

The business receives

• Historical Data

• Monthly Incremental Data

• Real-Time Transactions

• Late Arriving Data

The objective is to build a scalable Medallion Architecture using Delta Lake that ingests, transforms, validates and serves analytics-ready data.

## Architecture

```text
                +----------------------+
                |      Source Data     |
                |----------------------|
                | Historical Orders    |
                | Incremental Orders   |
                | Streaming Transactions|
                | Late Arriving Orders |
                +----------+-----------+
                           |
                           v
                +----------------------+
                |     Bronze Layer     |
                | Raw Delta Tables     |
                +----------+-----------+
                           |
                           v
                +----------------------+
                |     Silver Layer     |
                | Clean & Enriched     |
                +----------+-----------+
                           |
                           v
                +----------------------+
                |      Gold Layer      |
                | Analytics & KPIs     |
                +----------+-----------+
```

## Technology Stack

| Technology | Purpose |
|------------|----------|
| **Databricks Free Edition** | Cloud-based development environment for building and running the data pipeline |
| **PySpark** | Distributed data processing, transformations, joins, and DataFrame operations |
| **Spark SQL** | SQL-based analytics and reporting on processed data |
| **Delta Lake** | ACID-compliant storage layer supporting MERGE, schema enforcement, and time travel |
| **Auto Loader (cloudFiles)** | Incremental streaming ingestion of transaction files |
| **Unity Catalog** | Centralized data governance and storage management using Volumes |
| **Delta MERGE** | Handles incremental updates and late-arriving records efficiently |
| **CSV Files** | Source data for batch, incremental, and streaming ingestion |
| **Git & GitHub** | Version control and project repository management |
| **Medallion Architecture** | Organizes data into Bronze, Silver, and Gold layers for scalable data engineering |

## Project Folder Structure

retailstream_data/

│

├── data/

│      customers.csv

│      products.csv

│      stores.csv

│

├── batch_initial/

├── batch_incremental/

├── late_arriving/

├── autoloader_landing/

│

├── delta/

│      bronze/

│      silver/

│      gold/

│

└── checkpoints/

## Medallion Architecture

### Bronze Layer

- Stores raw source data without business transformations.
- Preserves original records for auditing and traceability.
- Adds audit columns such as ingestion timestamp and source file.

### Silver Layer

- Cleans and validates raw data.
- Removes inconsistencies and enriches records using dimension     tables.
- Calculates business metrics such as revenue and margin.

### Gold Layer

- Contains aggregated business reports and KPIs.
- Optimized for dashboards and business analytics.
- Serves as the reporting layer for Power BI.

In [0]:
# Import Required Libraries

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

from delta.tables import DeltaTable

from datetime import datetime

import time

# Notebook Parameters

Databricks Widgets allow users to execute the notebook dynamically by selecting different execution modes without modifying the source code.

This makes the pipeline reusable and production-ready.

In [0]:

# Databricks Widgets


dbutils.widgets.dropdown(
    "load_mode",
    "overwrite",
    ["overwrite", "append"]
)

dbutils.widgets.dropdown(
    "pipeline_layer",
    "All",
    ["Bronze", "Silver", "Gold", "All"]
)

dbutils.widgets.dropdown(
    "environment",
    "Development",
    ["Development", "Testing", "Production"]
)

In [0]:
dbutils.widgets.text(
    "batch_file",
    "orders_2024_01.csv",
    "Batch File"
)

In [0]:
LOAD_MODE = dbutils.widgets.get("load_mode")
PIPELINE_LAYER = dbutils.widgets.get("pipeline_layer")
ENVIRONMENT = dbutils.widgets.get("environment")
batch_file = dbutils.widgets.get("batch_file")

In [0]:
print(f"Environment : {ENVIRONMENT}")
print(f"Load Mode   : {LOAD_MODE}")
print(f"Layer       : {PIPELINE_LAYER}")

Environment : Development
Load Mode   : overwrite
Layer       : All


# Project Configuration

This section initializes all project-level configurations, including storage paths, Delta Lake locations, checkpoint directories, and reusable variables.

Using centralized configuration improves maintainability and makes the notebook easier to deploy across different environments (Development, Testing, Production).

In [0]:
CATALOG = "workspace"

SCHEMA = "default"

VOLUME = "retailstream_data"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

In [0]:
DATA_PATH = f"{BASE_PATH}/data"

BATCH_INITIAL_PATH = f"{DATA_PATH}/batch_initial"

BATCH_INCREMENTAL_PATH = f"{DATA_PATH}/batch_incremental"

LATE_DATA_PATH = f"{DATA_PATH}/late_arriving"

AUTOLOADER_PATH = f"{DATA_PATH}/autoloader_landing"

In [0]:
BRONZE_PATH = f"{BASE_PATH}/delta/bronze"

SILVER_PATH = f"{BASE_PATH}/delta/silver"

GOLD_PATH = f"{BASE_PATH}/delta/gold"

In [0]:
BRONZE_ORDERS_PATH = f"{BRONZE_PATH}/bronze_orders"

BRONZE_TRANSACTIONS_PATH = f"{BRONZE_PATH}/bronze_transactions"

SILVER_ORDERS_PATH = f"{SILVER_PATH}/silver_orders"

GOLD_MONTHLY_SALES_PATH = f"{GOLD_PATH}/gold_monthly_sales"

GOLD_PAYMENT_SUMMARY_PATH = f"{GOLD_PATH}/gold_payment_summary"

GOLD_CATEGORY_SALES_PATH = f"{GOLD_PATH}/gold_sales_by_category"

GOLD_REGION_SALES_PATH = f"{GOLD_PATH}/gold_sales_by_region"

GOLD_TOP_PRODUCTS_PATH = f"{GOLD_PATH}/gold_top_products"

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints"

SCHEMA_LOCATION = f"{CHECKPOINT_PATH}/schema/bronze_transactions"

TRANSACTION_CHECKPOINT = f"{CHECKPOINT_PATH}/bronze_transactions"

# Global Variables

Global variables are defined once and reused throughout the notebook to avoid code duplication and improve readability.

In [0]:
PROJECT_NAME = "RetailStream Inc."

PIPELINE_NAME = "Retail Data Engineering Pipeline"

CURRENT_YEAR = 2026

# Utility Functions

Helper functions are created to minimize repetitive code, improve readability, and simplify pipeline maintenance.

In [0]:
# Read CSV Function

def read_csv(path, infer_schema=True):
    """
    Reads a CSV file from the given path.
    """

    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", infer_schema)
        .csv(path)
    )

In [0]:
# Read Delta Table

def read_delta(path):

    return (
        spark.read
        .format("delta")
        .load(path)
    )

In [0]:
# Write Delta Table

def write_delta(df, path, mode):

    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .save(path)
    )

In [0]:
def display_summary(df, table_name):

    print("="*50)

    print(f"Dataset : {table_name}")

    print(f"Rows : {df.count()}")

    print(f"Columns : {len(df.columns)}")

    display(df.limit(5))

    print("="*50)

In [0]:
from pyspark.sql.functions import col

def duplicate_count(df, column):
    return (
        df.groupBy(column)
          .count()
          .filter(col("count") > 1)
          .count()
    )

# Pipeline Logging

Logging is an essential component of production-grade data pipelines. It helps monitor execution, track progress, and simplify debugging by recording important events throughout the pipeline execution.

## Project Workflow

Landing Files

│

├── Historical Orders

├── Incremental Orders

├── Streaming Transactions

└── Late Arriving Orders

↓

Bronze Layer

Raw Delta Tables

↓

Silver Layer

Business Ready Clean Data

↓

Gold Layer

Aggregated Reports

In [0]:
from datetime import datetime

def log_info(message):
    print(f"[INFO] {datetime.now()} | {message}")

def log_warning(message):
    print(f"[WARNING] {datetime.now()} | {message}")
def log_error(message):
    print(f"[ERROR] {datetime.now()} | {message}")

# Data Validation

Data quality checks are performed after each major processing stage to ensure data consistency and reliability before moving to the next layer.

In [0]:
from pyspark.sql.functions import col

def check_duplicates(df, column_name):
    duplicate_count = (
        df.groupBy(column_name)
          .count()
          .filter(col("count") > 1)
          .count()
    )

    print(f"Duplicate Records ({column_name}) : {duplicate_count}")

    return duplicate_count

In [0]:
# Null Check
def check_nulls(df):
    print("Null Value Summary")

    for column in df.columns:
        count = df.filter(col(column).isNull()).count()
        print(f"{column} : {count}")

In [0]:
# Record Count Validation

def validate_count(df, expected_count):
    actual = df.count()

    print(f"Expected : {expected_count}")
    print(f"Actual   : {actual}")

    if actual == expected_count:
        print("Validation Passed")
    else:
        print("Validation Failed")

In [0]:
# Schema Validation

def display_schema(df):
    print("Schema")

    df.printSchema()

# Exception Handling

Critical pipeline operations are wrapped in exception handling blocks to improve reliability and simplify troubleshooting.

# Pipeline Execution Monitoring

The total execution time of the pipeline is recorded for performance monitoring purposes.

# Bronze Layer

The Bronze layer stores raw data exactly as received from multiple sources with only minimal transformations and audit columns.

## Task 1 – Historical Batch Load

### Objective

This task ingests the historical January order dataset into the Bronze Layer.

The pipeline performs the following operations:

- Reads the historical CSV file.
- Adds audit columns (`ingestion_timestamp`, `source_file`).
- Writes data into a Delta table.
- Performs data validation.
- Displays execution summary.

In [0]:
log_info("Starting Historical Batch Load")

[INFO] 2026-07-12 10:45:33.769603 | Starting Historical Batch Load


In [0]:
orders_df = read_csv(
    f"{BATCH_INITIAL_PATH}/{batch_file}"
)

In [0]:
display_schema(orders_df)

Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
validate_count(orders_df,20)

Expected : 20
Actual   : 20
Validation Passed


In [0]:
orders_bronze_df = (
    orders_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit(batch_file))
)

In [0]:
display_summary(
    orders_bronze_df,
    "Historical Orders"
)

Dataset : Historical Orders
Rows : 20
Columns : 10


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD001,C026,P008,1,28000,2024-01-11,S02,DELIVERED,2026-07-12T10:45:37.532Z,orders_2024_01.csv
ORD002,C021,P004,4,18000,2024-01-12,S03,DELIVERED,2026-07-12T10:45:37.532Z,orders_2024_01.csv
ORD003,C020,P003,2,50000,2024-01-04,S01,DELIVERED,2026-07-12T10:45:37.532Z,orders_2024_01.csv
ORD004,C031,P007,1,18000,2024-01-09,S03,DELIVERED,2026-07-12T10:45:37.532Z,orders_2024_01.csv
ORD005,C005,P004,3,800,2024-01-08,S03,DELIVERED,2026-07-12T10:45:37.532Z,orders_2024_01.csv


In [0]:
write_delta(
    orders_bronze_df,
    BRONZE_ORDERS_PATH,
    LOAD_MODE
)

In [0]:
bronze_orders = read_delta(
    BRONZE_ORDERS_PATH
)
display_summary(
    bronze_orders,
    "Bronze Orders"
)

Dataset : Bronze Orders
Rows : 20
Columns : 10


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD001,C026,P008,1,28000,2024-01-11,S02,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv
ORD002,C021,P004,4,18000,2024-01-12,S03,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv
ORD003,C020,P003,2,50000,2024-01-04,S01,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv
ORD004,C031,P007,1,18000,2024-01-09,S03,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv
ORD005,C005,P004,3,800,2024-01-08,S03,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv


In [0]:
check_duplicates(
    bronze_orders,
    "order_id"
)

check_nulls(
    bronze_orders
)

Duplicate Records (order_id) : 0
Null Value Summary
order_id : 0
customer_id : 0
product_id : 0
quantity : 0
unit_price : 0
order_date : 0
store_id : 0
status : 0
ingestion_timestamp : 0
source_file : 0


In [0]:
assert bronze_orders.count()==20

In [0]:

log_info("Bronze Orders Created Successfully")

[INFO] 2026-07-12 10:45:47.544808 | Bronze Orders Created Successfully


## Task 2

### Incremental Batch Load

In this step, the February order file is processed. Existing records are identified and removed before appending only the new orders to the Bronze layer.

In [0]:
log_info("Starting Incremental Batch Load")

[INFO] 2026-07-12 10:45:47.766407 | Starting Incremental Batch Load


In [0]:
incremental_df = read_csv(
    f"{BATCH_INCREMENTAL_PATH}/orders_2024_02.csv"
)

In [0]:
display_summary(
    incremental_df,
    "Incremental Orders"
)

Dataset : Incremental Orders
Rows : 22
Columns : 8


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status
ORD003,C020,P003,2,50000,2024-01-04,S01,DELIVERED
ORD013,C029,P005,1,4000,2024-01-12,S02,DELIVERED
ORD021,C005,P003,3,1500,2024-02-09,S03,DELIVERED
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED
ORD040,C030,P002,4,18000,2024-02-15,S03,DELIVERED


In [0]:
validate_count(
    incremental_df,
    20
)

Expected : 20
Actual   : 22
Validation Failed


In [0]:
bronze_orders = read_delta(
    BRONZE_ORDERS_PATH
)

In [0]:
new_orders = (
    incremental_df.alias("inc")
    .join(
        bronze_orders.select("order_id").alias("br"),
        on="order_id",
        how="left_anti"
    )
)

In [0]:
display_summary(
    new_orders,
    "New Incremental Orders"
)

Dataset : New Incremental Orders
Rows : 20
Columns : 8


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status
ORD021,C005,P003,3,1500,2024-02-09,S03,DELIVERED
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED
ORD040,C030,P002,4,18000,2024-02-15,S03,DELIVERED
ORD034,C013,P008,3,50000,2024-02-16,S02,DELIVERED
ORD026,C021,P003,4,18000,2024-02-06,S03,DELIVERED


In [0]:
print(f"New Orders : {new_orders.count()}")

New Orders : 20


In [0]:
new_orders = (
    new_orders
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("orders_2024_02.csv"))
)

In [0]:
write_delta(
    new_orders,
    BRONZE_ORDERS_PATH,
    "append"
)

In [0]:
bronze_orders = read_delta(
    BRONZE_ORDERS_PATH
)
display_summary(
    bronze_orders,
    "Bronze Orders After Incremental Load"
)

Dataset : Bronze Orders After Incremental Load
Rows : 40
Columns : 10


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD021,C005,P003,3,1500,2024-02-09,S03,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv
ORD040,C030,P002,4,18000,2024-02-15,S03,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv
ORD034,C013,P008,3,50000,2024-02-16,S02,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv
ORD026,C021,P003,4,18000,2024-02-06,S03,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv


In [0]:
check_duplicates(
    bronze_orders,
    "order_id"
)

check_nulls(
    bronze_orders
)

Duplicate Records (order_id) : 0
Null Value Summary
order_id : 0
customer_id : 0
product_id : 0
quantity : 0
unit_price : 0
order_date : 0
store_id : 0
status : 0
ingestion_timestamp : 0
source_file : 0


In [0]:
print(f"Total Bronze Records : {bronze_orders.count()}")

Total Bronze Records : 40


In [0]:

log_info("Incremental Batch Load Completed Successfully")


[INFO] 2026-07-12 10:46:03.379540 | Incremental Batch Load Completed Successfully


In [0]:
assert bronze_orders.count() == 40

## Task 3 – Real-Time Streaming using Auto Loader

### Objective

This task simulates real-time transaction ingestion using Databricks Auto Loader.

The pipeline:

- Continuously monitors the landing folder.
- Automatically detects newly arrived transaction files.
- Infers schema and stores metadata.
- Writes streaming data into the Bronze Delta table.
- Performs validation after ingestion.

In [0]:
log_info("Starting Auto Loader Streaming Pipeline")

[INFO] 2026-07-12 10:46:04.388437 | Starting Auto Loader Streaming Pipeline


In [0]:
transactions_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option(
         "cloudFiles.schemaLocation",
          SCHEMA_LOCATION
)

.load(AUTOLOADER_PATH)
)

In [0]:
display_schema(transactions_stream)

Schema
root
 |-- txn_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- txn_timestamp: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- gateway_status: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
stream_query = (
    transactions_stream.writeStream
        .format("delta")
        .option(
    "checkpointLocation",
    TRANSACTION_CHECKPOINT
)

.option(
    "path",
    BRONZE_TRANSACTIONS_PATH
)
        .trigger(availableNow=True)
        .start()
)

In [0]:
stream_query.awaitTermination()

In [0]:
log_info("Streaming Query Finished Successfully")

[INFO] 2026-07-12 10:46:08.903957 | Streaming Query Finished Successfully


In [0]:
bronze_transactions = read_delta(
    BRONZE_TRANSACTIONS_PATH
)

In [0]:
print(f"Total Transactions : {bronze_transactions.count()}")

Total Transactions : 15


In [0]:
check_duplicates(
    bronze_transactions,
    "txn_id"
)

check_nulls(
    bronze_transactions
)

log_info(
    "Streaming Load Completed Successfully"
)

Duplicate Records (txn_id) : 0
Null Value Summary
txn_id : 0
order_id : 0
payment_method : 0
amount : 0
txn_timestamp : 0
currency : 0
gateway_status : 0
_rescued_data : 15
[INFO] 2026-07-12 10:46:13.759565 | Streaming Load Completed Successfully


In [0]:
assert bronze_transactions.count() > 0

log_info(
    f"Total Transactions Loaded : {bronze_transactions.count()}"
)

[INFO] 2026-07-12 10:46:14.465025 | Total Transactions Loaded : 15


## Task 4 – Late Arriving Data using Delta MERGE

### Objective

This task processes late-arriving January orders that were not available during the initial historical load.

The pipeline:

- Reads late-arriving records.
- Adds audit columns.
- Performs an idempotent Delta MERGE.
- Inserts only new records.
- Validates the final Bronze table.

In [0]:
log_info("Starting Late Arriving Data Processing")

late_orders_df = read_csv(
    f"{LATE_DATA_PATH}/orders_2024_01_LATE.csv"
)

[INFO] 2026-07-12 10:46:14.706158 | Starting Late Arriving Data Processing


In [0]:
display_summary(
    late_orders_df,
    "Late Arriving Orders"
)

Dataset : Late Arriving Orders
Rows : 5
Columns : 8


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status
ORD_L01,C004,P003,4,4000,2024-01-15,S04,DELIVERED
ORD_L02,C002,P005,1,18000,2024-01-15,S04,DELIVERED
ORD_L03,C023,P005,4,2000,2024-01-19,S04,DELIVERED
ORD_L04,C017,P008,3,2000,2024-01-10,S04,DELIVERED
ORD_L05,C030,P001,1,4000,2024-01-09,S04,DELIVERED


In [0]:
late_orders_df = (
    late_orders_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("orders_2024_01_LATE.csv"))
)

In [0]:
bronze_delta = DeltaTable.forPath(
    spark,
    BRONZE_ORDERS_PATH
)

In [0]:
log_info(
"Executing Delta MERGE Operation"
)

[INFO] 2026-07-12 10:46:17.247141 | Executing Delta MERGE Operation


In [0]:
(
bronze_delta.alias("target")
.merge(
late_orders_df.alias("source"),
"target.order_id = source.order_id"
)
.whenNotMatchedInsertAll()
.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
log_info(
"MERGE Operation Completed Successfully"
)

[INFO] 2026-07-12 10:46:20.883833 | MERGE Operation Completed Successfully


In [0]:
bronze_orders = read_delta(
BRONZE_ORDERS_PATH
)

display_summary(
bronze_orders,
"Bronze Orders After MERGE"
)

display(
bronze_orders
.filter(col("store_id")=="S04")
)

Dataset : Bronze Orders After MERGE
Rows : 45
Columns : 10


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD_L01,C004,P003,4,4000,2024-01-15,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L02,C002,P005,1,18000,2024-01-15,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L03,C023,P005,4,2000,2024-01-19,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L04,C017,P008,3,2000,2024-01-10,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L05,C030,P001,1,4000,2024-01-09,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv


order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv
ORD027,C003,P003,4,18000,2024-02-17,S04,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv
ORD038,C016,P005,3,28000,2024-02-23,S04,DELIVERED,2026-07-12T10:45:55.166Z,orders_2024_02.csv
ORD_L01,C004,P003,4,4000,2024-01-15,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L02,C002,P005,1,18000,2024-01-15,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L03,C023,P005,4,2000,2024-01-19,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L04,C017,P008,3,2000,2024-01-10,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv
ORD_L05,C030,P001,1,4000,2024-01-09,S04,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv


In [0]:
s04_count = (
    bronze_orders
    .filter(col("store_id")=="S04")
    .count()
)

log_info(
f"Late Arriving Records Inserted : {s04_count}"
)

[INFO] 2026-07-12 10:46:23.653039 | Late Arriving Records Inserted : 8


In [0]:
validate_count(
bronze_orders,
45
)

Expected : 45
Actual   : 45
Validation Passed


In [0]:
log_info(
"Late Arriving Data MERGE Completed Successfully"
)

[INFO] 2026-07-12 10:46:24.427794 | Late Arriving Data MERGE Completed Successfully


In [0]:
check_duplicates(
bronze_orders,
"order_id"
)

check_nulls(
bronze_orders
)

assert bronze_orders.count()==45

Duplicate Records (order_id) : 0
Null Value Summary
order_id : 0
customer_id : 0
product_id : 0
quantity : 0
unit_price : 0
order_date : 0
store_id : 0
status : 0
ingestion_timestamp : 0
source_file : 0


# Silver Layer

The Silver layer cleans, enriches and joins business entities.

## Task 5 – Silver Layer (Cleaned & Enriched Data)

### Objective

This task transforms the Bronze Orders dataset into a clean and business-ready Silver dataset.

The pipeline:

- Reads Bronze Orders and dimension tables.
- Joins customer, product and store information.
- Computes revenue and margin.
- Removes unnecessary foreign keys.
- Writes the enriched dataset to the Silver Delta layer.

In [0]:
customers_df = read_csv(
    f"{DATA_PATH}/customers.csv"
)

In [0]:
products_df = read_csv(
    f"{DATA_PATH}/products.csv"
)

In [0]:
stores_df = read_csv(
    f"{DATA_PATH}/stores.csv"
)

In [0]:
# Read Bronze Orders
bronze_orders = read_delta(
BRONZE_ORDERS_PATH
)

In [0]:
# Rename duplicate column before joining
stores_df = stores_df.withColumnRenamed("city", "store_city")

log_info(
"Joining Customer, Product and Store Dimensions"
)

# Join all tables
silver_df = (
    bronze_orders
    .join(customers_df, on="customer_id", how="left")
    .join(products_df, on="product_id", how="left")
    .join(stores_df, on="store_id", how="left")
)

# Business Calculations
silver_df = (
    silver_df
    .withColumn("revenue", col("quantity") * col("unit_price"))


    .withColumn("total_cost", col("quantity") * col("cost_price"))
    .withColumn("margin", col("revenue") - col("total_cost"))
)

log_info(
"Business Metrics Calculated Successfully"
)

[INFO] 2026-07-12 10:46:33.190579 | Joining Customer, Product and Store Dimensions
[INFO] 2026-07-12 10:46:33.191880 | Business Metrics Calculated Successfully


In [0]:
display_schema(
silver_df
)

Schema
root
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cost_price: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- total_cost: integer (nullable = true)
 |-- margin: integer (nullable = true)



In [0]:
display_summary(
silver_df,
"Silver Orders"
)

Dataset : Silver Orders
Rows : 45
Columns : 23


store_id,product_id,customer_id,order_id,quantity,unit_price,order_date,status,ingestion_timestamp,source_file,customer_name,city,tier,product_name,category,brand,cost_price,store_name,region,store_city,revenue,total_cost,margin
S04,P003,C004,ORD_L01,4,4000,2024-01-15,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv,Customer_4,Bangalore,SILVER,Headphones,Accessories,SoundMax,1500,Ahmedabad West,West,Ahmedabad,16000,6000,10000
S04,P005,C002,ORD_L02,1,18000,2024-01-15,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv,Customer_2,Bangalore,GOLD,Keyboard,Accessories,TypeRight,1200,Ahmedabad West,West,Ahmedabad,18000,1200,16800
S04,P005,C023,ORD_L03,4,2000,2024-01-19,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv,Customer_23,Mumbai,BRONZE,Keyboard,Accessories,TypeRight,1200,Ahmedabad West,West,Ahmedabad,8000,4800,3200
S04,P008,C017,ORD_L04,3,2000,2024-01-10,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv,Customer_17,Pune,GOLD,Smartwatch,Accessories,TimeSmart,3000,Ahmedabad West,West,Ahmedabad,6000,9000,-3000
S04,P001,C030,ORD_L05,1,4000,2024-01-09,DELIVERED,2026-07-12T10:46:18.003Z,orders_2024_01_LATE.csv,Customer_30,Bangalore,BRONZE,Laptop,Electronics,TechPro,45000,Ahmedabad West,West,Ahmedabad,4000,45000,-41000


In [0]:
check_nulls(
silver_df
)

print("Duplicate Order Validation")

print(
    silver_df.count()
    -
    silver_df.dropDuplicates(["order_id"]).count()
)

Null Value Summary
store_id : 0
product_id : 0
customer_id : 0
order_id : 0
quantity : 0
unit_price : 0
order_date : 0
status : 0
ingestion_timestamp : 0
source_file : 0
customer_name : 0
city : 0
tier : 0
product_name : 0
category : 0
brand : 0
cost_price : 0
store_name : 0
region : 0
store_city : 0
revenue : 0
total_cost : 0
margin : 0
Duplicate Order Validation
0


In [0]:
print("Row Count")

print(silver_df.count())

Row Count
45


In [0]:
write_delta(
silver_df,
SILVER_ORDERS_PATH,
"overwrite"
)

In [0]:
log_info(
"Silver Layer Written Successfully"
)

[INFO] 2026-07-12 10:46:58.335851 | Silver Layer Written Successfully


In [0]:
silver_orders = read_delta(
SILVER_ORDERS_PATH
)

display_summary(
silver_orders,
"Silver Orders"
)

validate_count(
silver_orders,
45
)

check_duplicates(
silver_orders,
"order_id"
)

log_info(
"Silver Layer Created Successfully"
)

Dataset : Silver Orders
Rows : 45
Columns : 23


store_id,product_id,customer_id,order_id,quantity,unit_price,order_date,status,ingestion_timestamp,source_file,customer_name,city,tier,product_name,category,brand,cost_price,store_name,region,store_city,revenue,total_cost,margin
S02,P008,C026,ORD001,1,28000,2024-01-11,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv,Customer_26,Bangalore,SILVER,Smartwatch,Accessories,TimeSmart,3000,Bangalore Hub,South,Bangalore,28000,3000,25000
S03,P004,C021,ORD002,4,18000,2024-01-12,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv,Customer_21,Bangalore,BRONZE,Monitor,Electronics,VisionPlus,8000,Delhi NCR,North,Delhi,72000,32000,40000
S01,P003,C020,ORD003,2,50000,2024-01-04,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv,Customer_20,Delhi,BRONZE,Headphones,Accessories,SoundMax,1500,Mumbai Central,West,Mumbai,100000,3000,97000
S03,P007,C031,ORD004,1,18000,2024-01-09,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv,Customer_31,Delhi,GOLD,Tablet,Electronics,TabLite,15000,Delhi NCR,North,Delhi,18000,15000,3000
S03,P004,C005,ORD005,3,800,2024-01-08,DELIVERED,2026-07-12T10:45:39.227Z,orders_2024_01.csv,Customer_5,Bangalore,GOLD,Monitor,Electronics,VisionPlus,8000,Delhi NCR,North,Delhi,2400,24000,-21600


Expected : 45
Actual   : 45
Validation Passed
Duplicate Records (order_id) : 0
[INFO] 2026-07-12 10:47:00.732682 | Silver Layer Created Successfully


In [0]:
assert silver_orders.count()==45

# Gold Layer

The Gold layer contains analytics-ready business reports.

## Task 6 – Gold Layer (Business Analytics using Spark SQL)

### Objective

This task generates business-ready analytical datasets from the Silver and Bronze layers using Spark SQL.

The reports created are:

- Monthly Sales Summary
- Payment Method Summary
- Sales by Category
- Sales by Region
- Top 5 Products

These datasets are stored in the Gold layer for reporting and dashboarding.

In [0]:
# Read Silver Layer
log_info("Starting Gold Layer Processing")

silver_orders = read_delta(
    SILVER_ORDERS_PATH
)

[INFO] 2026-07-12 10:47:01.862013 | Starting Gold Layer Processing


In [0]:
silver_orders.createOrReplaceTempView(
"silver_orders"
)

In [0]:
# Monthly Sales Summary
monthly_sales = spark.sql("""

SELECT

date_format(order_date,'yyyy-MM') as month,

COUNT(order_id) as total_orders,

ROUND(SUM(revenue),2) as total_revenue,

ROUND(SUM(margin),2) as total_margin,

ROUND(AVG(revenue),2) as avg_order_value

FROM silver_orders

GROUP BY month

ORDER BY month

""")

In [0]:
display(monthly_sales)

month,total_orders,total_revenue,total_margin,avg_order_value
2024-01,25,731900,71700,29276.0
2024-02,20,959000,615800,47950.0


In [0]:
write_delta(
    monthly_sales,
    GOLD_MONTHLY_SALES_PATH,
    "overwrite"
)

In [0]:
# Sales by Category
sales_by_category = (
    silver_orders
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_sales"),
        sum("margin").alias("total_margin"),
        sum("quantity").alias("total_quantity")
    )
)

In [0]:
display(sales_by_category)

category,total_sales,total_margin,total_quantity
Electronics,839900,-59100,53
Accessories,851000,746600,62


In [0]:
write_delta(
    sales_by_category,
    GOLD_CATEGORY_SALES_PATH,
    "overwrite"
)

In [0]:
gold_monthly_sales = read_delta(
    GOLD_MONTHLY_SALES_PATH
)

gold_sales_by_category = read_delta(
    GOLD_CATEGORY_SALES_PATH
)

In [0]:
sales_by_region = (
    silver_orders
    .groupBy("region")
    .agg(
        sum("revenue").alias("total_sales"),
        sum("margin").alias("total_margin")
    )
)

display(sales_by_region)

(
sales_by_region.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema","true")
.save(f"{GOLD_PATH}/gold_sales_by_region")
)

region,total_sales,total_margin
North,444900,122000
South,576000,240200
West,670000,325300


In [0]:
top_products = (
    silver_orders
    .groupBy("product_name")
    .agg(
        sum("revenue").alias("total_sales")
    )
    .orderBy(desc("total_sales"))
    .limit(5)
)

In [0]:

gold_top_products = (
    spark.read
    .format("delta")
    .load(f"{GOLD_PATH}/gold_top_products")
)
display(top_products)

product_name,total_sales
Tablet,369500
Headphones,355000
Smartwatch,308000
Smartphone,186000
Monitor,180400


In [0]:
write_delta(
    top_products,
    GOLD_TOP_PRODUCTS_PATH,
    "overwrite"
)

In [0]:
gold_top_products = read_delta(
    GOLD_TOP_PRODUCTS_PATH
)

In [0]:
display(gold_top_products)

product_name,total_sales
Tablet,369500
Headphones,355000
Smartwatch,308000
Smartphone,186000
Monitor,180400


In [0]:
log_info("RetailStream Pipeline Executed Successfully")

print(f"Bronze Orders        : {bronze_orders.count()}")

print(f"Transactions         : {bronze_transactions.count()}")

print(f"Silver Orders        : {silver_orders.count()}")

print(f"Monthly Sales        : {gold_monthly_sales.count()}")

print(f"Category Sales       : {gold_sales_by_category.count()}")

print(f"Top Products         : {gold_top_products.count()}")

[INFO] 2026-07-12 10:47:16.671202 | RetailStream Pipeline Executed Successfully
Bronze Orders        : 45
Transactions         : 15
Silver Orders        : 45
Monthly Sales        : 2
Category Sales       : 2
Top Products         : 5


## Project Validation

- Bronze Layer Created Successfully

- Silver Layer Created Successfully

- Gold Layer Created Successfully

- Streaming Data Processed

- Incremental Load Successful

- Late Arriving Data Successfully Merged

In [0]:
assert gold_monthly_sales.count() > 0

assert gold_sales_by_category.count() > 0

assert gold_top_products.count() > 0

## Conclusion

This project successfully implements an end-to-end RetailStream Data Engineering Pipeline using PySpark, Delta Lake and Databricks.

The pipeline processes historical batch data, incremental data, streaming transactions and late arriving records using a Medallion Architecture (Bronze, Silver and Gold).

The final Gold Layer produces analytics-ready datasets that can be directly connected to Power BI for business reporting and decision making.